### Transformer training pipeline

Dataset

↓

Tokenization

↓

Vocabulary creation

↓

Encoding sentences → numbers

↓

Padding

↓

Transformer model(Encoder+Decoder)

↓

Loss (CrossEntropy)

↓

Training loop

↓

Prediction

In [2]:
# step-1 install libraries 
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset     # load_dataset is a function from Hugging Face's datasets library. Its job is to load datasets from various sources.
from torch.nn.utils.rnn import pad_sequence
import math


In [9]:
# step-2 Load HuggingFace dataset
 
# It's a dataset identifier with two parts:
# First part (opus_books): The dataset name/collection
# Second part (en-fr): The specific language pair (English→French)

# Load the "opus_books" dataset for English-French translation
# "opus_books" is a collection of book translations
# "en-fr" means English-French language pair

dataset = load_dataset("opus_books", "en-fr")
# This creates a DatasetDict with 'train' split containing English-French sentence pairs
# Each example looks like: {'translation': {'en': 'Hello', 'fr': 'Bonjour'}}
# dataset is a Dictionary with keys: ["train"]

# Take only first 5000 examples from training set (to keep things manageable)
# .select(range(5000)) picks rows with indices 0 to 4999

data = dataset["train"].select(range(5000))
#      ↑            ↑      ↑
#      |            |      └── Take first 5000 rows
#      |            └── Access the 'train' split
#      └── DatasetDict containing all splits
# Now 'data' has 5000 examples instead of the full dataset (which could be lakhs)

# Extract English sentences from each example
# [x["translation"]["en"] for x in data] means:
# For each item 'x' in 'data', go to x["translation"] then get ["en"] value
# .lower() converts all text to lowercase (for consistency)
input = [x["translation"]["en"].lower() for x in data]
#        ↑  ↑            ↑
#        |  |            └── Get 'en' (English) from translation
#        |  └── Get 'translation' dictionary from the row
#        └── Each item 'x' is one row from data

# Result: List of English sentences like ["hello", "how are you", "the cat", ...]

# Extract French translations similarly
# Same process but get ["fr"] value instead of "en"
targets = [x["translation"]["fr"].lower() for x in data]
# Result: List of French sentences like ["bonjour", "comment allez-vous", "le chat", ...]



In [7]:
print(dataset)
first=dataset["train"][1]
print(first)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
{'id': '1', 'translation': {'en': 'Alain-Fournier', 'fr': 'Alain-Fournier'}}


In [ ]:
# step-3 Build Vocabulary

# Define a function that creates a word-to-number mapping from sentences
def build_vocab(sentences):
    
    # Initialize vocabulary with 3 special tokens:
    # <pad> (0): Used to make all sentences same length by padding
    # <start> (1): Marks beginning of sentence for decoder
    # <end> (2): Marks end of sentence for decoder
    
    vocab = {"<pad>": 0, "<start>": 1, "<end>": 2}
    
    # Start numbering new words from index 3 (since 0,1,2 are taken)
    idx = 3
    
    # Loop through each sentence in our list
    for sent in sentences:
        # Split sentence into individual words (by spaces)
        for word in sent.split():
            # If this word is not already in vocabulary
            if word not in vocab:
                # Add word to vocabulary with current index number
                vocab[word] = idx
                # Increment index for next new word
                idx += 1
    
    # Return the complete vocabulary dictionary
    return vocab

# Build vocabulary for English sentences (source language)
# input = list of English sentences

eng_vocab = build_vocab(input)
# Result: {"<pad>":0, "<start>":1, "<end>":2, "hello":3, "world":4, ...}
# Build vocabulary for French sentences (target language)
# targets = list of French sentences

tgt_vocab = build_vocab(targets)
# Result: {"<pad>":0, "<start>":1, "<end>":2, "bonjour":3, "monde":4, ...}
# Create reverse vocabulary for English (number → word)
# This swaps keys and values from eng_vocab

eng_inv = {v: k for k, v in eng_vocab.items()}
# Result: {0:"<pad>", 1:"<start>", 2:"<end>", 3:"hello", 4:"world", ...}
# .items() gives you pairs, v:k swaps them - so numbers become keys and words become values!
# Create reverse vocabulary for French (number → word)

tgt_inv = {v: k for k, v in tgt_vocab.items()}
# Result: {0:"<pad>", 1:"<start>", 2:"<end>", 3:"bonjour", 4:"monde", ...}

# Get vocabulary sizes (number of unique words including special tokens)
eng_vocab_size = len(eng_vocab)  # Used for embedding layer dimensions
tgt_vocab_size = len(tgt_vocab)  # Used for output layer dimensions

In [ ]:
# step-4 

def encode(sentence,vocab):
    
# Takes a sentence (like "hello world") and vocabulary dictionary
# Returns a tensor (list of numbers) representing the sentence

   return torch.tensor([vocab[x] for x in sentence.split() if x in vocab])
   # 1. sentence.split() → ["hello", "world"]
   # 2. vocab[w] for each word → [3, 4] (if hello=3, world=4 in vocab)
   # 3. torch.tensor() → converts to PyTorch tensor
   
eng_data=[encode(s,eng_vocab) for s in input]
# For each English sentence in 'inputs', convert to numbers
# Result: List of tensors, one per sentence
# After encoding:
#src_data = [
    #tensor([3, 4]),        # "hello world"
    #tensor([5, 6, 7])      # "how are you"]

tgt_input = []   # Will store decoder INPUT sequences (with <start>)
tgt_output = []  # Will store decoder OUTPUT sequences (with <end>)

french_tokens=[encode(s,tgt_vocab) for s in targets]
# Result: List of tensors like [tensor([3,4]), tensor([5,6,7,8]), ...]
# Example: "bonjour monde" → tensor([3,4])

for tokens in french_tokens:
    
    tgt_input.append(torch.cat([torch.tensor([tgt_vocab["<start>"]]), tokens]))
    # tokens is already a tensor from encode() function
    # Decoder input: <start> + tokens
    # torch.cat() joins tensors together
    # cat = concatenate (join/combine) tensors together
    #a = torch.tensor([1, 2, 3])
    #b = torch.tensor([4, 5, 6])
    #c = torch.cat([a, b])
    #print(c)  # tensor([1, 2, 3, 4, 5, 6])
    # Just puts them one after another
    
    tgt_output.append(torch.cat([tokens, torch.tensor([tgt_vocab["<end>"]])]))
    # Decoder output: tokens + <end>
    # Result: tensor([3, 4, 2]) for "bonjour monde"
    


### simple note 1:

French sentence: "bonjour monde"

                          ↓

                   tokens = [3, 4]

                    ↓              ↓

        tgt_input = [1, 3, 4]    tgt_output = [3, 4, 2]

        (to decoder)              (what decoder should predict)

Position:    1       2       3

Input:     <start>  bonjour  monde

Target:    bonjour  monde    <end>

In [18]:
# step-5 Padding

from torch.nn.utils.rnn import pad_sequence

# pad_sequence-A PyTorch function that takes a list of tensors of different lengths and pads them to all have the same length.
# Takes list of tensors of different lengths
# Returns a single tensor with all sequences padded to same length

eng_pad = pad_sequence(eng_data,batch_first=True,padding_value=0)
tgt_in_pad = pad_sequence(tgt_input,batch_first=True,padding_value=0)
tgt_out_pad = pad_sequence(tgt_output,batch_first=True,padding_value=0)

# Your data
# tgt_input = [
    # tensor([1, 3, 4]),        # "<start> bonjour monde" (len 3)
    # tensor([1, 5, 6, 7])      # "<start> comment allez vous" (len 4)

# pad_sequence finds max_len = 4
# tgt_in_pad = pad_sequence(tgt_input, batch_first=True, padding_value=0)

# Result:
# [
#     [1, 3, 4, 0],    # padded with 0 at the end
#     [1, 5, 6, 7]     # unchanged
# ]

# result = pad_sequence(sequences, batch_first=True, padding_value=0)

# Without batch_first: shape [max_len, batch_size]
# [
#     [3, 5, 8],
#     [4, 6, 9],
#     [0, 7, 10],
#     [0, 0, 11]
# ]

# With batch_first=True: shape [batch_size, max_len] (easier to work with)
# [
#     [3, 4, 0, 0],
#     [5, 6, 7, 0],
#     [8, 9, 10, 11]
# ]

print(eng_pad.shape)
print(tgt_in_pad.shape)
print(tgt_out_pad.shape)

torch.Size([5000, 200])
torch.Size([5000, 120])
torch.Size([5000, 120])


In [19]:
# step-6 Position embedding

class PositionalEncoding(nn.Module):
    # This class adds position information to word embeddings
    # Because transformers process all words in parallel and don't know word order
    # Inherits from nn.Module (PyTorch's base class)
    
    
    def __init__(self, d_model, max_len=500):
        
        # CONSTRUCTOR METHOD
        # __init__ is the CONSTRUCTOR - it runs ONCE when creating a new object
        # It sets up all the LAYERS and PARAMETERS that this block will use
        
        super().__init__()   # Call parent class (nn.Module) constructor
        # d_model = embedding dimension (e.g., 8, 512)
        # max_len = maximum sentence length we'll ever see (default 500) 
    
        pe = torch.zeros(max_len,d_model) # Shape: [10, 6]   
        # Position 1: [0, 0, 0, 0, 0, 0]
        # Position 2: [0, 0, 0, 0, 0, 0]
        # ...
        # Position 10:[0, 0, 0, 0, 0, 0]

        
        pos = torch.arange(0,max_len).unsqueeze(1)
        # torch.arange(0,10) → [0,1,2,3,4,5,6,7,8,9] a list
        # unsqueeze(1) → [[0],[1],[2],...[9]]  Shape: [max_len,1]  
        # unsqueeze(1)**: Reshapes the vector from a 1D array to a 2D column matrix (Shape: `[max_len, 1]`).   
        
        div = torch.exp(torch.arange(0,d_model,2)*(-math.log(10000)/d_model)) 
        # PE(pos, 2i) = sin(pos / 10000^(2i / d_model))
        # PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))

        # 1/10000^(2i/d_model)  = 10000^(-2i/d_model)
                      # = exp(log(10000^(-2i/d_model)))
                      # = exp(-2i/d_model × log(10000))
                      # = exp(2i × (-log(10000)/d_model))
                      # 2i is exactly torch.arange(0, d_model, 2)  [0,2,4...]   when i=0, 2i=0 ;  i=1, 2i=2 .....
        
    
        pe[:, 0::2] = torch.sin(pos * div)  # Even columns get sine
        
        # position * div_term: [10,1] × [1,3] broadcasts to [10,3]
        # sin() applied to each element

        pe[:,1::2]=torch.cos(position*div_term)
        # Fill odd columns (1,3,5) with cosine
        # pe[:, 1::2] means: ALL rows (:), columns starting at 1, step 2 (1,3,5)
        # Same position * div_term but with cos()
        
    
        self.pe = pe.unsqueeze(0)
        
        #Before unsqueeze(0):
        #pe = torch.zeros(10, 6)  # Shape: [10, 6] - 10 positions, 6 features
        # [pos1: [0,0,0,0,0,0]
        #  pos2: [0,0,0,0,0,0]
        #  ...
        #  pos10:[0,0,0,0,0,0]]

        #After unsqueeze(0):
        #self.pe = pe.unsqueeze(0)  # Shape: [1, 10, 6]
        # Adds batch dimension at front
    
    def forward(self,x):
        
        # FORWARD METHOD 
        # forward defines what happens when data PASSES THROUGH this block
        # model = PositionalEncoding(d_model=8)  # Create instance
        # When we do this:
        # x_with_pos = model(x)  # ← This calls forward() automatically!
        
        x=x+self.pe[:,:x.size(1)].to(device)
        
        # Input x shape: [2, 3, 6]  (batch=2 sentences, 3 words each, 6 features)
        #x.size(1)  # = 3 (sequence length)

        # self.pe shape: [1, 10, 6]
        # self.pe[:, :3]  # Take first 3 positions from all batches
        
        return x

